In [1]:
import os
from pathlib import Path

PROJ_ROOT = Path(os.getcwd())
print(PROJ_ROOT)
os.chdir(str(PROJ_ROOT.parent))

print(os.getcwd())

/home/Anton/audio-stem-splitter/notebooks
/home/Anton/audio-stem-splitter


# Импорт необходимых библиотек и функций

In [2]:
import torch
from model.chunking import separate_source
from model.hdemucs import hdemucs, device
import librosa
import torch
import soundfile as sf

# Конфигурация сегментирования и загрузки ресурсов

In [3]:
ASSETS_DIR = "assets"
TRACK = "test_audio.mp3"
SAVE_DIR = "save"

segment = 10
overlap_proportion = 0.1
overlap = segment * overlap_proportion

# Загрузка аудиофайла и модели на вычислительное устройство, нормализация трека

In [4]:
audio, sample_rate = librosa.load("assets/test_audio.mp3", sr=44100, mono=False)
sample_waveform = torch.tensor(audio)  
hdemucs = hdemucs.to(device)

ref = sample_waveform.mean(0)
sample_waveform = (sample_waveform - ref.mean()) / ref.std()

sample_waveform = sample_waveform.to(device=device)

/home/Anton/audio-stem-splitter/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Получение стемов, сохранение полученных стемов в аудиофайлы формата .wav

In [6]:
sources = separate_source(hdemucs, torch.unsqueeze(sample_waveform, 0), segment, overlap, sample_rate, device=device).squeeze(0)

ref = ref.to(device=device)
sources = sources * ref.std() + ref.mean()

sources = sources.cpu()
source_names = ["drums", "bass", "other", "vocals"]

if not os.path.exists(f"{os.getcwd()}/{SAVE_DIR}"):
    os.mkdir(f"{os.getcwd()}/{SAVE_DIR}")

for i, name in enumerate(source_names):
    stem = sources[i] 
    
    filename = f"{SAVE_DIR}/{name}.wav"

    sf.write(filename, stem.cpu().numpy().T, sample_rate)
    print(f"Файл {filename} сохранен.")


Файл save/drums.wav сохранен.
Файл save/bass.wav сохранен.
Файл save/other.wav сохранен.
Файл save/vocals.wav сохранен.


# Сохранение результата в формате zip-архива

In [ ]:
ZIP_NAME = (TRACK.split(".")[0]) + 

import shutil
shutil.make_archive()